# AutoML を使用した損傷した自動車部品の特定

## 概要
Agent Platform（Vertex AI）は、ML を構築するための Google Cloud サービスを 1 つの統合された UI と API のもとに集約しています。Agent Platform では、[AutoML](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/beginner/beginners-guide) またはカスタムコードトレーニングを使用してモデルを簡単にトレーニングおよび比較でき、すべてのモデルが 1 つの中央モデルリポジトリに保存されます。これらのモデルは、Agent Platform 上の同じエンドポイントにデプロイできます。

AutoML Vision を使用すると、機械学習（ML）の専門知識があまりない人でも、高品質な画像分類モデルをトレーニングできます。この実践的なラボでは、損傷した自動車部品を自動的に認識するカスタム ML モデルを作成する方法を学びます。

ML モデルを作成すると、すぐに利用可能になります。UI または REST API を使用して、Google Cloud Console から直接予測の生成を開始できます。

## 学習目標
このラボでは、以下の方法を学びます:
- CSV ファイルを使用してラベル付きデータセットを Cloud Storage にアップロードし、Managed Dataset（管理対象データセット）として Agent Platform に接続する
- アップロードされた画像を確認し、データセットに誤りがないことを確認する
- AutoML トレーニングジョブを開始する
- トレーニング済みモデルを確認し、その精度を評価する
- トレーニング済みモデルをデプロイし、オンライン予測を試す


## セットアップ


In [ ]:
import base64
import csv
import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from google.cloud import aiplatform
from google.cloud.aiplatform.gapic.schema import predict

## 訓練用画像を Cloud Storage にアップロードする
このタスクでは、使用する訓練用画像を Cloud Storage にアップロードします。これにより、後でデータを Agent Platform にインポートしやすくなります。

損傷した自動車部品の画像を分類するモデルをトレーニングするには、ラベル付けされた訓練データをマシンに提供する必要があります。モデルはこのデータを使用して各画像を理解し、自動車の部品や損傷のある部品を区別できるようになります。

この例では、モデルは次の 5 種類の損傷した自動車部品を分類することを学習します: **bumper**（バンパー）、**engine compartment**（エンジンルーム）、**hood**（ボンネット/フード）、**lateral**（側面）、**windshield**（フロントガラス）。


### Cloud Storage バケットの作成
まず、環境変数を設定するために以下のコードを実行します:



In [ ]:
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]
BUCKET = PROJECT_ID

次に、Cloud Storage バケットを作成するために、以下のコマンドを実行します:


In [ ]:
!gcloud storage ls --buckets | grep -w gs://{BUCKET}/ || gcloud storage buckets create --project={PROJECT_ID} \
    --default-storage-class=standard \
    --location=us-central1 \
    gs://{BUCKET}

### 自動車画像を Storage バケットにアップロードする
訓練用画像は公開の Cloud Storage バケットで提供されています。

画像を自身の Cloud Storage バケットにコピーするには、以下のコマンドを実行します:



In [ ]:
!gcloud storage cp -r gs://asl-public-data/data/car_damage_lab_images* gs://{BUCKET}/car_damage_lab_images

サンプル画像をいくつか確認してみましょう。


In [ ]:
!mkdir ./car_damage_lab_images

LABELS = ["bumper", "engine_compartment", "hood", "lateral", "windshield"]
for label in LABELS:
    !gcloud storage cp gs://{BUCKET}/car_damage_lab_images/{label}/{label}1.jpg ./car_damage_lab_images
    img = mpimg.imread(f"car_damage_lab_images/{label}1.jpg")
    plt.imshow(img)
    plt.title(label)
    plt.show()

!rm -rf car_damage_lab_images

## データセットの作成
このタスクでは、新しいデータセットを作成し、Agent Platform がアクセスできるように訓練用画像にデータセットを接続します。

各行に訓練用画像への URL とそれに関連付けられたラベルを含む CSV ファイルを作成します。<br>
また、オプションとして `TRAIN` や `VALIDATION` などのデータ分割を表す列を追加することもできます。<br>

```
TRAIN, gs://bucket/img1.jpg, bumper
VALIDATION, gs://bucket/img2.jpg, hood
```

CSV 形式の詳細については、[公式ドキュメント](https://cloud.google.com/vision/automl/object-detection/docs/csv-format)で確認できます。

今回のケースでは、各画像のラベルはディレクトリ名として表されています。そのため、次のコマンドで CSV を作成できます。


In [ ]:
CSV_FILE = "car_damage_dataset.csv"

data_list = []

for label in LABELS:
    data = !gcloud storage ls gs://{BUCKET}/car_damage_lab_images/{label}
    for i, img_path in enumerate(data):
        assign = "TRAIN" if i % 5 > 0 else "VALIDATION"  # 80:20
        data_list.append([assign, img_path, label])

with open(CSV_FILE, "w") as f:
    write = csv.writer(f)
    write.writerows(data_list)

In [ ]:
!head ./car_damage_dataset.csv

この CSV ファイルを GCS バケットにアップロードしましょう。


In [ ]:
!gcloud storage cp ./{CSV_FILE} gs://{BUCKET}/car_damage_lab_images

## 管理対象データセット（Managed Dataset）の作成

作成した CSV ファイルを指定してデータをインポートしましょう。

**演習**: [API ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/1.19.0/google.cloud.aiplatform.ImageDataset#google_cloud_aiplatform_ImageDataset_create)を参考に、以下の `aiplatform.ImageDataset.create` メソッドのパラメータを定義してください。<br>
ここでは以下を定義してください:
- `display_name`: データセットの表示名。
- `gcs_source`: 作成した CSV データセットの GCS ファイルパス。
- `import_schema_uri`: データセットのスキーマ。スキーマは[こちら](https://github.com/googleapis/python-aiplatform/blob/d06b22d1ac6197c460092739e8572b9beb08bd63/google/cloud/aiplatform/schema.py#L48-L53)で確認でき、`aiplatform.schema.dataset.ioformat.image.<Image Schema Name>` と記述して選択します。今回のデータセットは単一ラベル分類用であることに注意してください。


In [ ]:
display_name = "damaged_car_parts"

ds = aiplatform.ImageDataset.create(
    # TODO: パラメータを定義
)

**注**: 画像がインポートされ、カテゴリと関連付けられるまでに約 9 〜 12 分かかります。次のステップに進む前に、この手順が完了するまで待つ必要があります。


## （省略可）画像の検査
アップロードジョブが完了したら、Agent Platform コンソールでデータセットを確認できます。

[コンソール](https://console.cloud.google.com) に移動し、`Agent Platform` -> `Models` > `Datasets` -> `damaged_car_parts` を開きます。

>![image](https://user-images.githubusercontent.com/6895245/175039729-f40ff31f-0a16-439a-97c3-fb29defbf16c.png)


## 画像ラベルの確認

1. **Filter labels** の下で任意のラベルをクリックして、特定の訓練用画像を表示します。*（例: engine_compartment）*

>![image](https://user-images.githubusercontent.com/6895245/175039977-61e7fcba-8750-4ac2-b8dd-50417f93c72c.png)


**注: 本番用のモデルを構築する場合は、高い精度を確保するためにラベルあたり少なくとも 100 枚の画像を用意することが推奨されます。今回はデモであるため、モデルを迅速にトレーニングできるよう各タイプ 20 枚の画像のみを使用しています。**


2. 画像のラベルが間違っている場合は、画像をクリックして正しいラベルを選択するか、トレーニングセットから画像を削除できます:

>![image](https://user-images.githubusercontent.com/6895245/175040279-b82b7a0b-fabb-4db0-ab95-20c9ec1c7b06.png)</kbd>


3. 次に、**Analyze** タブをクリックしてラベルごとの画像数を表示します。ブラウザの右側に [Label Stats] ウィンドウが表示されます。

>![image](https://user-images.githubusercontent.com/6895245/175040916-4d9cdc3b-f59e-4005-a5a1-9d11d49ad518.png)


**注**: データセットのラベル付けの支援が必要な場合は、Agent Platform Labeling Services を使用することで、人間のラベラーと協力して精度の高いラベルを生成できます。


## モデルのトレーニング
これでモデルのトレーニングを開始する準備が整いました！

Agent Platform AutoML は、機械学習モデルを自動的に設計・構築するサービスです。
そのため、モデルのコードを記述する必要はありません。代わりに、Python SDK を使用してトレーニングを開始しましょう。

トレーニングを実行する際、保有している予算に基づいてノード時間を指定できます。
画像分類ジョブの場合、最小ノード時間は 8 です。

**演習**: [API ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/1.19.1/google.cloud.aiplatform.AutoMLImageTrainingJob)を参考に、以下の `aiplatform.AutoMLImageTrainingJob` コンストラクタのパラメータを定義してください。<br>
ここでは以下を定義してください:
- `display_name`: トレーニングジョブの表示名。
- `model_type`: モデルタイプ。クラウドでのトレーニングには `"CLOUD"` を指定します。
- `prediction_type`: モデルの予測タイプ。今回のデータセットは分類用（classification）であることに注意してください。
- `multi_label`: モデルがマルチラベル分類用であるかを指定するブール値。今回のデータセットは単一ラベル分類用であることに注意してください。


In [ ]:
job = aiplatform.AutoMLImageTrainingJob(
    # TODO: パラメータを定義
)

In [ ]:
model_display_name = "damaged_car_parts_model"

model = job.run(
    dataset=ds,
    model_display_name=model_display_name,
    budget_milli_node_hours=8 * 1000,  # 8,000 ミリノード時間（8 ノード時間）
    sync=True,
)

**注**: モデルのトレーニングが完了するまでに約 30 分かかります。


## モデルの評価
このタスクでは、モデルを評価します。分類問題であるため、多くの分類指標を確認できます。

1. `Agent Platform` -> `Model` -> `damaged_car_parts_model` から新しくトレーニングしたモデルに移動します。**Evaluate** タブをクリックします。このタブには、モデルの Precision（適合率）と Recall（再現率）に関する情報が表示されます。以下のように表示されるはずです:

>![image](https://user-images.githubusercontent.com/6895245/175043316-ccad262e-ac16-444f-a4bc-dcf96975bf4e.png)


2. **Confidence threshold**（信頼度のしきい値）スライダーを調整して、その影響を確認することもできます。

3. 下にスクロールして **Confusion matrix**（混同行列）を表示します。

>![image](https://user-images.githubusercontent.com/6895245/175046016-4cba9dc2-8040-4dc8-b6cf-2566856f5ab8.png)

このセクションには、モデルの精度を評価し、トレーニングデータの改善すべき領域を特定するのに役立つ一般的な機械学習メトリクスが用意されています。


## エンドポイントへのデプロイ

モデルをデプロイし、API 経由で予測を呼び出してみましょう。

[`model.deploy` 関数](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_deploy)は、モデルを Agent Platform のエンドポイントにデプロイします。

既存の Agent Platform エンドポイントを指定することもできますが、設定されていない場合は新しいエンドポイントが自動的に作成されます。

**注**: デプロイには最大 10 分かかる場合があります。

**演習**: [API ドキュメント](https://cloud.google.com/python/docs/reference/aiplatform/1.19.0/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_deploy)を参考に、以下の `aiplatform.Model.deploy` メソッドのパラメータを定義してください。<br>
ここでは以下を定義してください:
- `display_name`: デプロイされたモデルの表示名。
- `min_replica_count` および `max_replica_count`: サーバーの最小および最大レプリカ数。現時点ではスケーリングを無効にするため、両方に `1` を指定します。
- `accelerator_type` および `accelerator_count`: 使用するアクセラレータのタイプと数。現時点では両方に `None` を指定します。


In [ ]:
endpoint = model.deploy(
    # TODO: パラメータを定義
)

## 予測の生成
いよいよ最も重要なステップです。モデルがこれまで見たことのないデータを使用して、トレーニング済みモデルで予測を生成します。

予測を生成する方法はいくつかあります。このラボでは、Python コマンドを使用してデプロイされたモデルを呼び出し、モデルが以下の画像をどのように分類するかを確認します:

- 損傷したバンパー（Damaged bumper）

- 損傷したエンジンルーム（Damaged engine compartment）

- ボンネットとバンパーの両方の損傷（Damage to the hood and the bumper。注: この最後の画像はマルチラベル分類問題の好例です）


In [ ]:
# テストデータは ../data ディレクトリに保存されています。
data_dir = "../data"
test_data = !ls {data_dir}/

for d in test_data:
    img = mpimg.imread(f"{data_dir}/{d}")
    plt.imshow(img)
    plt.title(d)
    plt.show()

Agent Platform にデプロイされたモデルを呼び出すには、API 経由で送信できるように画像を base64 形式にエンコードする必要があります。


**演習**: 予測を呼び出す前に画像データを base64 にエンコードする行を追加して、以下の予測 API 呼び出しコードを完成させてください。

参考:
- [base64.b64encode](https://docs.python.org/3/library/base64.html#base64.b64encode)
- [bytes.decode](https://docs.python.org/3/library/stdtypes.html#bytes)


In [ ]:
for image in test_data:
    with open(f"../data/{image}", "rb") as f:
        file_content = f.read()

    # base64 にエンコードし、ImageClassificationPredictionInstance にラップ
    # TODO: 画像を base64 エンコードする行を追加してください
    # encoded_content =
    instance = predict.instance.ImageClassificationPredictionInstance(
        content=encoded_content
    ).to_value()

    # エンドポイントを呼び出す
    response = endpoint.predict([instance])

    # レスポンスから情報を抽出
    prediction = response.predictions[0]
    argmax = np.argmax(prediction["confidences"])
    confidence = prediction["confidences"][argmax]
    predicted_label = prediction["displayNames"][argmax]

    # 可視化して出力
    img = mpimg.imread(f"../data/{image}")
    plt.imshow(img)
    plt.title(f"prediction: {predicted_label} ({confidence})")
    plt.show()
    for label, confidence in zip(
        prediction["displayNames"], prediction["confidences"]
    ):
        print(f"{label:20}: {confidence:7f}")
    print("*" * 50)

モデルの予測結果はどうでしたか？3 つの画像すべてを正しく予測できましたか？

## まとめ
このラボでは、独自のカスタム機械学習モデルをトレーニングし、予測を生成する方法を学びました。訓練用画像を Cloud Storage にアップロードし、Agent Platform がこれらの画像を検索できるように CSV ファイルを使用しました。ラベル付き画像を検査して誤りがないか確認し、トレーニング済みモデルを評価しました。これで、独自の画像データセットでモデルをトレーニングするために必要な知識が身につきました。


# License

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.